# Практика «Технологии ИИ». Полносвязная нейронная сеть

Классификация рукописных цифр (набор `digits`, 8×8, 10 классов).

- архитектура — многослойный перцептрон, все слои полносвязные;
- активация скрытых слоёв — **ReLU**;
- на выходе — **softmax**;
- функция потерь — категориальная **кросс-энтропия**;
- ниже строятся графики loss и accuracy на обучающей и тестовой выборках.


## 1. Данные

Пиксели лежат в диапазоне 0…16, поэтому делим на 16 и получаем значения от 0 до 1. Метки переводим в one-hot: кросс-энтропия сравнивает распределение softmax с вектором, где у верного класса стоит 1.


In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

# На macOS BLAS внутри NumPy иногда пишет ложное предупреждение на обычном matmul.
warnings.filterwarnings("ignore", message=".*encountered in matmul.*")

digits = load_digits()
X = digits.data.astype(np.float64) / 16.0
y = digits.target.astype(np.int64)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

def one_hot(labels, n_classes=10):
    encoded = np.zeros((labels.shape[0], n_classes), dtype=np.float64)
    encoded[np.arange(labels.shape[0]), labels] = 1.0
    return encoded

Y_train = one_hot(y_train)
Y_test = one_hot(y_test)

print(f"обучающая выборка: {X_train.shape[0]} объектов, {X_train.shape[1]} признаков")
print(f"тестовая выборка:  {X_test.shape[0]} объектов")
print(f"классы: {np.unique(y).tolist()}")


In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(10, 3))
for ax, image, label in zip(axes.ravel(), digits.images, digits.target):
    ax.imshow(image, cmap="gray_r")
    ax.set_title(str(label))
    ax.axis("off")
fig.suptitle("Примеры цифр из выборки")
plt.tight_layout()
plt.show()


## 2. ReLU, softmax и кросс-энтропия

Скрытые слои:

$$
\mathrm{ReLU}(z) = \max(0, z)
$$

Выходной слой переводит логиты в вероятности классов:

$$
\mathrm{softmax}(z)_k = \frac{e^{z_k}}{\sum_j e^{z_j}}
$$

Перед экспонентой вычитается максимум логита, чтобы не получить переполнение.

Категориальная кросс-энтропия, $y$ — one-hot, $p$ — softmax:

$$
L = -\frac{1}{N}\sum_{i=1}^{N}\sum_{k=1}^{C} y_{ik}\log p_{ik}
$$

Для пары softmax + кросс-энтропия градиент по логитам выхода равен $(p - y) / N$. Дальше он проходит по цепному правилу через ReLU: производная ReLU равна 1 при $z > 0$ и 0 иначе.


In [ ]:
def relu(z):
    return np.maximum(0.0, z)


def relu_derivative(z):
    return (z > 0.0).astype(np.float64)


def softmax(logits):
    shifted = logits - np.max(logits, axis=1, keepdims=True)
    exp_values = np.exp(shifted)
    return exp_values / np.sum(exp_values, axis=1, keepdims=True)


def cross_entropy(probabilities, y_one_hot):
    clipped = np.clip(probabilities, 1e-12, 1.0)
    return -np.mean(np.sum(y_one_hot * np.log(clipped), axis=1))


## 3. Полносвязная сеть

Каждый слой считает $z = a W + b$.

- после скрытых слоёв применяется ReLU;
- после последнего слоя — softmax.

Веса инициализируются методом Хе ($\mathcal{N}(0, \sqrt{2/n_{\mathrm{in}}})$): так ReLU не затухает на старте. Обучение — мини-батчевый градиентный спуск.


In [ ]:
class FullyConnectedNetwork:
    def __init__(self, layer_sizes, seed=42):
        rng = np.random.default_rng(seed)
        self.weights = []
        self.biases = []
        for n_in, n_out in zip(layer_sizes[:-1], layer_sizes[1:]):
            self.weights.append(rng.normal(0.0, np.sqrt(2.0 / n_in), size=(n_in, n_out)))
            self.biases.append(np.zeros((1, n_out)))
        self._cache = None

    def forward(self, X):
        activations = [X]
        pre_activations = []
        current = X
        last = len(self.weights) - 1
        for index, (weight, bias) in enumerate(zip(self.weights, self.biases)):
            z = current @ weight + bias
            pre_activations.append(z)
            current = softmax(z) if index == last else relu(z)
            activations.append(current)
        self._cache = (activations, pre_activations)
        return current

    def backward(self, y_one_hot):
        activations, pre_activations = self._cache
        batch_size = y_one_hot.shape[0]
        # dL/dz на выходе softmax + кросс-энтропия
        delta = (activations[-1] - y_one_hot) / batch_size
        grad_w = [None] * len(self.weights)
        grad_b = [None] * len(self.biases)
        for index in reversed(range(len(self.weights))):
            grad_w[index] = activations[index].T @ delta
            grad_b[index] = np.sum(delta, axis=0, keepdims=True)
            if index > 0:
                delta = (delta @ self.weights[index].T) * relu_derivative(pre_activations[index - 1])
        return grad_w, grad_b

    def update(self, grad_w, grad_b, learning_rate):
        for index in range(len(self.weights)):
            self.weights[index] -= learning_rate * grad_w[index]
            self.biases[index] -= learning_rate * grad_b[index]

    def predict(self, X):
        return np.argmax(self.forward(X), axis=1)

    def evaluate(self, X, y_labels, y_one_hot):
        probabilities = self.forward(X)
        loss_value = cross_entropy(probabilities, y_one_hot)
        accuracy = np.mean(np.argmax(probabilities, axis=1) == y_labels)
        return loss_value, accuracy


model = FullyConnectedNetwork([64, 128, 64, 10])
print("слои:", [weight.shape[1] for weight in model.weights])
print("число параметров:", sum(w.size + b.size for w, b in zip(model.weights, model.biases)))


## 4. Обучение

На каждой эпохе выборка перемешивается, сеть обновляется по батчам, затем считаются loss и accuracy на всей обучающей и тестовой выборках.


In [ ]:
def train(model, X_train, Y_train, y_train, X_test, Y_test, y_test,
         epochs=40, batch_size=64, learning_rate=0.15, seed=42):
    rng = np.random.default_rng(seed)
    history = {key: [] for key in ("train_loss", "test_loss", "train_acc", "test_acc")}
    n_samples = X_train.shape[0]

    for epoch in range(1, epochs + 1):
        order = rng.permutation(n_samples)
        for start in range(0, n_samples, batch_size):
            batch = order[start:start + batch_size]
            model.forward(X_train[batch])
            grad_w, grad_b = model.backward(Y_train[batch])
            model.update(grad_w, grad_b, learning_rate)

        train_loss, train_acc = model.evaluate(X_train, y_train, Y_train)
        test_loss, test_acc = model.evaluate(X_test, y_test, Y_test)
        history["train_loss"].append(train_loss)
        history["test_loss"].append(test_loss)
        history["train_acc"].append(train_acc)
        history["test_acc"].append(test_acc)

        if epoch == 1 or epoch % 5 == 0 or epoch == epochs:
            print(
                f"эпоха {epoch:2d}  "
                f"train loss {train_loss:.4f}  acc {train_acc:.3f}  |  "
                f"test loss {test_loss:.4f}  acc {test_acc:.3f}"
            )
    return history


history = train(model, X_train, Y_train, y_train, X_test, Y_test, y_test)


## 5. Графики loss и accuracy


In [ ]:
epochs_axis = np.arange(1, len(history["train_loss"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(epochs_axis, history["train_loss"], label="train")
axes[0].plot(epochs_axis, history["test_loss"], label="test")
axes[0].set_title("Кросс-энтропия")
axes[0].set_xlabel("Эпоха")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_axis, history["train_acc"], label="train")
axes[1].plot(epochs_axis, history["test_acc"], label="test")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Эпоха")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0.0, 1.02)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

final_train_loss, final_train_acc = model.evaluate(X_train, y_train, Y_train)
final_test_loss, final_test_acc = model.evaluate(X_test, y_test, Y_test)
print(f"итог train: loss {final_train_loss:.4f}, accuracy {final_train_acc:.3f}")
print(f"итог test:  loss {final_test_loss:.4f}, accuracy {final_test_acc:.3f}")


## 6. Примеры предсказаний на тесте


In [ ]:
rng = np.random.default_rng(0)
sample_ids = rng.choice(X_test.shape[0], size=12, replace=False)
predictions = model.predict(X_test[sample_ids])

fig, axes = plt.subplots(2, 6, figsize=(10, 4))
for ax, index, predicted in zip(axes.ravel(), sample_ids, predictions):
    image = (X_test[index] * 16).reshape(8, 8)
    actual = y_test[index]
    ax.imshow(image, cmap="gray_r")
    color = "green" if predicted == actual else "crimson"
    ax.set_title(f"y={actual}, ŷ={predicted}", color=color, fontsize=10)
    ax.axis("off")
fig.suptitle("Случайные объекты тестовой выборки")
plt.tight_layout()
plt.show()
